In [0]:
CREATE OR REPLACE TABLE production.supply_analytics.dst_cancellation_dataset AS

-- ============================================
-- PARAMETERS - UPDATE THESE
-- ============================================
WITH params AS (
  SELECT 
    DATE '2025-01-01' AS yoy_pre_start_date,
    DATE '2025-01-15' AS yoy_pre_end_date,
    DATE '2025-01-16' AS yoy_event_week_start,
    DATE '2025-01-30' AS yoy_post_end_date,
    DATE '2026-01-01' AS event_pre_start_date,
    DATE '2026-01-15' AS event_pre_end_date,
    DATE '2026-01-16' AS event_week_start_str,
    DATE '2026-01-30' AS event_post_end_date,
    ARRAY('Germany') AS countries,
    ARRAY() AS tour_ids
),

-- ============================================
-- Base cancellation data with period windows
-- ============================================
cancellation_base AS (
  SELECT
    fb.booking_id,
    fb.tour_id,
    tour.user_id AS supplier_id,
    dl.country_name,
    fb.nr,
    fb.gmv,
    fb.tickets,
    COALESCE(fb.status_id, -1) AS status_id,
    CAST(fb.date_of_checkout AS date) AS dt_checkout,
    CAST(fb.date_of_travel AS date) AS dt_travel,
    CASE
      WHEN CAST(fb.date_of_checkout AS date) BETWEEN params.yoy_pre_start_date AND params.yoy_pre_end_date THEN 'PRE_LY'
      WHEN CAST(fb.date_of_checkout AS date) BETWEEN params.yoy_event_week_start AND params.yoy_post_end_date THEN 'POST_LY'
      WHEN CAST(fb.date_of_checkout AS date) BETWEEN params.event_pre_start_date AND params.event_pre_end_date THEN 'PRE_CY'
      WHEN CAST(fb.date_of_checkout AS date) BETWEEN params.event_week_start_str AND params.event_post_end_date THEN 'POST_CY'
    END AS period_window,
    CASE
      WHEN CAST(fb.date_of_checkout AS date) BETWEEN params.yoy_pre_start_date AND params.yoy_post_end_date
        THEN params.yoy_event_week_start
      ELSE params.event_week_start_str
    END AS event_start
  FROM production.dwh.fact_booking_v2 AS fb
  CROSS JOIN params
  INNER JOIN production.dwh.dim_status AS s
    ON COALESCE(fb.status_id, -1) = s.status_id
  LEFT JOIN production.dwh.dim_tour AS tour
    ON fb.tour_id = tour.tour_id
  LEFT JOIN production.dwh.dim_location AS dl
    ON tour.location_id = dl.location_id
  WHERE
    s.status_display IN ('Active', 'Cancelled')
    AND (SIZE(params.countries) = 0 OR ARRAY_CONTAINS(params.countries, dl.country_name))
    AND (
      CAST(fb.date_of_checkout AS date) BETWEEN params.yoy_pre_start_date AND params.yoy_post_end_date
      OR CAST(fb.date_of_checkout AS date) BETWEEN params.event_pre_start_date AND params.event_post_end_date
    )
    AND (
      SIZE(params.tour_ids) = 0
      OR fb.tour_id IN (SELECT tour_id FROM params LATERAL VIEW explode(params.tour_ids) e AS tour_id)
    )
)

-- ============================================
-- Aggregate cancellation rates by country, period window,
-- and travel date horizon (3M, 6M, 12M from event start)
-- ============================================
SELECT
  country_name,
  period_window,

  -- ---- Overall (all travel dates) ----
  COUNT(DISTINCT booking_id)                                                    AS total_bookings,
  COUNT(DISTINCT CASE WHEN status_id = 2 THEN booking_id END)                  AS cancelled_bookings,
  COUNT(DISTINCT CASE WHEN status_id = 2 THEN booking_id END)
    / NULLIF(COUNT(DISTINCT booking_id), 0)                                     AS cancellation_rate,
  SUM(nr)                                                                       AS total_nr,
  SUM(CASE WHEN status_id = 2 THEN nr ELSE 0 END)                              AS cancelled_nr,
  SUM(CASE WHEN status_id = 2 THEN nr ELSE 0 END)
    / NULLIF(SUM(nr), 0)                                                        AS cancellation_rate_nr,

  -- ---- 3M horizon: travel date within 3 months of event start ----
  COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 3)
    THEN booking_id END)                                                        AS bookings_3m,
  COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 3)
    AND status_id = 2 THEN booking_id END)                                      AS cancelled_bookings_3m,
  COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 3)
    AND status_id = 2 THEN booking_id END)
    / NULLIF(COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 3)
    THEN booking_id END), 0)                                                    AS cancellation_rate_3m,
  SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 3)
    THEN nr ELSE 0 END)                                                         AS nr_3m,
  SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 3)
    AND status_id = 2 THEN nr ELSE 0 END)                                       AS cancelled_nr_3m,
  SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 3)
    AND status_id = 2 THEN nr ELSE 0 END)
    / NULLIF(SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 3)
    THEN nr ELSE 0 END), 0)                                                     AS cancellation_rate_nr_3m,

  -- ---- 6M horizon: travel date within 6 months of event start ----
  COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 6)
    THEN booking_id END)                                                        AS bookings_6m,
  COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 6)
    AND status_id = 2 THEN booking_id END)                                      AS cancelled_bookings_6m,
  COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 6)
    AND status_id = 2 THEN booking_id END)
    / NULLIF(COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 6)
    THEN booking_id END), 0)                                                    AS cancellation_rate_6m,
  SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 6)
    THEN nr ELSE 0 END)                                                         AS nr_6m,
  SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 6)
    AND status_id = 2 THEN nr ELSE 0 END)                                       AS cancelled_nr_6m,
  SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 6)
    AND status_id = 2 THEN nr ELSE 0 END)
    / NULLIF(SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 6)
    THEN nr ELSE 0 END), 0)                                                     AS cancellation_rate_nr_6m,

  -- ---- 12M horizon: travel date within 12 months of event start ----
  COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 12)
    THEN booking_id END)                                                        AS bookings_12m,
  COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 12)
    AND status_id = 2 THEN booking_id END)                                      AS cancelled_bookings_12m,
  COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 12)
    AND status_id = 2 THEN booking_id END)
    / NULLIF(COUNT(DISTINCT CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 12)
    THEN booking_id END), 0)                                                    AS cancellation_rate_12m,
  SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 12)
    THEN nr ELSE 0 END)                                                         AS nr_12m,
  SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 12)
    AND status_id = 2 THEN nr ELSE 0 END)                                       AS cancelled_nr_12m,
  SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 12)
    AND status_id = 2 THEN nr ELSE 0 END)
    / NULLIF(SUM(CASE WHEN dt_travel BETWEEN event_start AND ADD_MONTHS(event_start, 12)
    THEN nr ELSE 0 END), 0)                                                     AS cancellation_rate_nr_12m,

  -- ---- Counts for context ----
  COUNT(DISTINCT tour_id)                                                       AS total_tours,
  COUNT(DISTINCT supplier_id)                                                   AS total_suppliers

FROM cancellation_base
WHERE period_window IS NOT NULL
GROUP BY country_name, period_window
ORDER BY
  country_name,
  CASE period_window
    WHEN 'PRE_LY'  THEN 1
    WHEN 'POST_LY' THEN 2
    WHEN 'PRE_CY'  THEN 3
    WHEN 'POST_CY' THEN 4
  END;
